# Week 6 – 7: Artificial Neural Network Implementation
## Android Permission Dataset — Benign vs Malware Classification

**Dataset:** [Kaggle — Android Permission Dataset](https://www.kaggle.com/datasets/saurabhshahane/android-permission-dataset)  
**Task:** Binary classification — `Class` column: `0 = Benign`, `1 = Malware`

---

## Step 0 — Setup: Imports & Configuration

In [ ]:
# ─── Core Libraries ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Sklearn Utilities ────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

# ─── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print(f'Pandas version     : {pd.__version__}')
print('Setup complete ✓')

---
## Step 1 — Load Data

In [ ]:
CSV_PATH = 'Android_Permission.csv'

df_raw = pd.read_csv(CSV_PATH)

print(f'Dataset shape : {df_raw.shape}  (rows × columns)')
print(f'\nTotal columns : {df_raw.shape[1]}')
print('\n── All Column Names ──────────────────────────────────────────')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  [{i:>3}] {col}')

In [ ]:
# Quick preview of the first few rows
df_raw.head(3)

In [ ]:
# Basic info
df_raw.info(verbose=False)

---
## Step 2 — Data Cleaning

In [ ]:
# ── 2a. Identify and drop non-numeric / irrelevant columns ────────────────────
NON_NUMERIC_COLS = ['App', 'Package', 'Category', 'Description', 'Related apps']

df = df_raw.drop(columns=NON_NUMERIC_COLS, errors='ignore').copy()
print(f'After dropping non-numeric text columns : {df.shape[1]} columns remain')

# ── 2b. Keep only numeric columns + target ────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df = df[numeric_cols].copy()
print(f'After keeping only numeric columns      : {df.shape[1]} columns remain')

# ── 2c. Drop constant / zero-variance columns (except target) ─────────────────
feature_cols_before = [c for c in df.columns if c != 'Class']
zero_var_cols = [c for c in feature_cols_before if df[c].nunique() <= 1]
df.drop(columns=zero_var_cols, inplace=True)
print(f'Zero-variance columns dropped           : {len(zero_var_cols)}')
print(f'Remaining columns                       : {df.shape[1]}')

# ── 2d. Handle missing values ─────────────────────────────────────────────────
df.fillna(0, inplace=True)
print(f'\nMissing value strategy : filled with 0 (permission absent)')

# ── 2e. Remove duplicates ─────────────────────────────────────────────────────
before_dup = df.shape[0]
df.drop_duplicates(inplace=True)
print(f'Duplicate rows removed                  : {before_dup - df.shape[0]}')
print(f'Final dataset shape                     : {df.shape}')

# ── 2f. Confirm binary target ─────────────────────────────────────────────────
print(f'\nUnique values in Class : {sorted(df["Class"].unique())}')
print(df['Class'].value_counts().rename({0: 'Benign (0)', 1: 'Malware (1)'}))

In [ ]:
# Cleaned column list
print('── Cleaned Column Names ──────────────────────────────────────')
for i, col in enumerate(df.columns, 1):
    print(f'  [{i:>3}] {col}')

In [ ]:
# ── Class Distribution Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts().sort_index()
labels = ['Benign (0)', 'Malware (1)']
colors = ['#4A90D9', '#E74C3C']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution — Count', fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=140, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Class Distribution — Proportion', fontweight='bold')

plt.suptitle('Benign vs Malware — Class Balance Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

imbalance_ratio = counts.iloc[0] / counts.iloc[1]
print(f'Imbalance ratio (Benign:Malware) ≈ {imbalance_ratio:.1f}:1')
print('→ Class weights will be used during training to handle imbalance.')

---
## Step 3 — Train / Test Split

In [ ]:
# Separate features and target
X = df.drop(columns=['Class']).values
y = df['Class'].values

INPUT_DIM = X.shape[1]
print(f'Number of input features (input layer dimensionality) : {INPUT_DIM}')
print(f'Total samples                                         : {len(y)}')

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'\nTraining set : {X_train.shape[0]} samples')
print(f'  Benign  : {(y_train==0).sum()}')
print(f'  Malware : {(y_train==1).sum()}')
print(f'\nTest set     : {X_test.shape[0]} samples')
print(f'  Benign  : {(y_test==0).sum()}')
print(f'  Malware : {(y_test==1).sum()}')

In [ ]:
# ── Prepare Normalized version (MinMaxScaler) ─────────────────────────────────
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm  = scaler.transform(X_test)

print('MinMaxScaler applied.')
print(f'  Train feature range: [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]')
print(f'  Test  feature range: [{X_test_norm.min():.2f},  {X_test_norm.max():.2f}]')

# Compute class weights for imbalance handling
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): w for c, w in zip(classes, cw)}
print(f'\nClass weights: {class_weight_dict}')

---
## Step 4 — Modeling

We run **8 experiments** total:

| # | Dataset | Model Architecture |
|---|---|---|
| 1 | Raw | Model 1: 128→64→32→16→8→1 |
| 2 | Raw | Model 2: 64→32→16→8→1 |
| 3 | Raw | Model 3: 32→16→8→1 |
| 4 | Raw | Model 4: 16→8→1 |
| 5 | Normalized | Model 1: 128→64→32→16→8→1 |
| 6 | Normalized | Model 2: 64→32→16→8→1 |
| 7 | Normalized | Model 3: 32→16→8→1 |
| 8 | Normalized | Model 4: 16→8→1 |

In [ ]:
# ── ANN Builder ───────────────────────────────────────────────────────────────
def build_ann(input_dim, layer_units):
    """
    Build a sequential ANN.
    layer_units: list where the last element is 1 (sigmoid output).
    Hidden layers use ReLU + BatchNorm + Dropout(0.2).
    """
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    
    for units in layer_units[:-1]:          # hidden layers
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(0.2))
    
    model.add(layers.Dense(1, activation='sigmoid'))   # output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


# ── Model configurations ──────────────────────────────────────────────────────
MODEL_CONFIGS = {
    'Model 1 (6L: 128-64-32-16-8-1)' : [128, 64, 32, 16, 8, 1],
    'Model 2 (5L: 64-32-16-8-1)'     : [64, 32, 16, 8, 1],
    'Model 3 (4L: 32-16-8-1)'        : [32, 16, 8, 1],
    'Model 4 (3L: 16-8-1)'           : [16, 8, 1],
}

EPOCHS     = 50
BATCH_SIZE = 256

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True, verbose=0
)

print('ANN builder ready.')
print(f'Epochs={EPOCHS}, Batch={BATCH_SIZE}, EarlyStopping patience=7')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MAIN EXPERIMENT LOOP — 8 experiments
# ══════════════════════════════════════════════════════════════════════════════

DATASETS = {
    'Raw'        : (X_train,      X_test),
    'Normalized' : (X_train_norm, X_test_norm),
}

results   = []
histories = {}
trained_models = {}

for dataset_name, (Xtr, Xte) in DATASETS.items():
    print(f'\n{"="*65}')
    print(f'  DATASET: {dataset_name}')
    print(f'{"="*65}')
    
    for model_name, layer_units in MODEL_CONFIGS.items():
        print(f'\n  Training {model_name} ...')
        
        tf.random.set_seed(SEED)
        model = build_ann(INPUT_DIM, layer_units)
        
        history = model.fit(
            Xtr, y_train,
            epochs           = EPOCHS,
            batch_size       = BATCH_SIZE,
            validation_split = 0.15,
            class_weight     = class_weight_dict,
            callbacks        = [early_stop],
            verbose          = 0
        )
        epochs_run = len(history.history['loss'])
        
        # Test evaluation
        y_pred_prob = model.predict(Xte, verbose=0).ravel()
        y_pred      = (y_pred_prob >= 0.5).astype(int)
        
        acc  = accuracy_score (y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score   (y_test, y_pred, zero_division=0)
        f1   = f1_score       (y_test, y_pred, zero_division=0)
        
        # Train evaluation
        y_tr_pred = (model.predict(Xtr, verbose=0).ravel() >= 0.5).astype(int)
        train_acc = accuracy_score(y_train, y_tr_pred)
        
        results.append({
            'Dataset'    : dataset_name,
            'Model'      : model_name,
            'Epochs Run' : epochs_run,
            'Train Acc'  : round(train_acc, 4),
            'Test Acc'   : round(acc,  4),
            'Precision'  : round(prec, 4),
            'Recall'     : round(rec,  4),
            'F1-Score'   : round(f1,   4),
        })
        
        key = f'{dataset_name}_{model_name}'
        histories[key]      = history.history
        trained_models[key] = (model, Xte)
        
        print(f'    Epochs={epochs_run} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}')

print('\nAll 8 experiments done!')

---
## Step 5 — Evaluation & Comparison

In [ ]:
# ── Results Table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)

print('='*95)
print('  ALL 8 EXPERIMENTS — RESULTS TABLE')
print('='*95)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', None)
display(results_df)

# Best model by F1-Score
best_idx = results_df['F1-Score'].idxmax()
best_row = results_df.loc[best_idx]

print(f'\n★  BEST MODEL by F1-Score  ★')
print(f'  Dataset   : {best_row["Dataset"]}')
print(f'  Model     : {best_row["Model"]}')
print(f'  Train Acc : {best_row["Train Acc"]:.4f}')
print(f'  Test Acc  : {best_row["Test Acc"]:.4f}')
print(f'  Precision : {best_row["Precision"]:.4f}')
print(f'  Recall    : {best_row["Recall"]:.4f}')
print(f'  F1-Score  : {best_row["F1-Score"]:.4f}')

In [ ]:
# ── Metric Bar Charts ─────────────────────────────────────────────────────────
metrics_to_plot = ['Test Acc', 'Precision', 'Recall', 'F1-Score']
model_labels    = [f'M{i+1}' for i in range(len(MODEL_CONFIGS))]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    raw_vals  = results_df[results_df['Dataset'] == 'Raw'       ][metric].values
    norm_vals = results_df[results_df['Dataset'] == 'Normalized'][metric].values
    
    x     = np.arange(len(model_labels))
    width = 0.35
    
    b1 = ax.bar(x - width/2, raw_vals,  width, label='Raw',        color='#4A90D9', alpha=0.85, edgecolor='white')
    b2 = ax.bar(x + width/2, norm_vals, width, label='Normalized', color='#E67E22', alpha=0.85, edgecolor='white')
    
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, fontsize=11)
    ax.set_ylim(0, 1.12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xlabel('Model', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    
    for b in list(b1) + list(b2):
        h = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('ANN Model Performance: Raw vs Normalized (M1-M4)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap of all metrics ────────────────────────────────────────────────────
pivot = results_df.copy()
pivot['Experiment'] = (
    pivot['Dataset'] + ' | ' +
    pivot['Model'].str.extract(r'(Model \d)')[0]
)
heat = pivot.set_index('Experiment')[['Train Acc','Test Acc','Precision','Recall','F1-Score']].astype(float)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heat, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0, annot_kws={'size': 10})
ax.set_title('Performance Heatmap — All 8 Experiments', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=20, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── Training History Plots ────────────────────────────────────────────────────
dataset_names = list(DATASETS.keys())
model_names   = list(MODEL_CONFIGS.keys())

fig, axes = plt.subplots(4, 4, figsize=(20, 16))

for di, dname in enumerate(dataset_names):
    for mi, mname in enumerate(model_names):
        key  = f'{dname}_{mname}'
        hist = histories[key]
        
        ax_acc  = axes[di*2    ][mi]
        ax_loss = axes[di*2 + 1][mi]
        
        ax_acc.plot(hist['accuracy'],     color='#4A90D9', label='Train')
        ax_acc.plot(hist['val_accuracy'], color='#E74C3C', linestyle='--', label='Val')
        ax_acc.set_title(f'{dname}\nM{mi+1}', fontsize=8, fontweight='bold')
        ax_acc.set_ylabel('Accuracy', fontsize=7)
        ax_acc.legend(fontsize=6)
        ax_acc.grid(alpha=0.3)
        
        ax_loss.plot(hist['loss'],     color='#4A90D9', label='Train')
        ax_loss.plot(hist['val_loss'], color='#E74C3C', linestyle='--', label='Val')
        ax_loss.set_ylabel('Loss', fontsize=7)
        ax_loss.set_xlabel('Epoch', fontsize=7)
        ax_loss.legend(fontsize=6)
        ax_loss.grid(alpha=0.3)

fig.suptitle('Training History — Accuracy & Loss (all 8 experiments)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix for Best Model ───────────────────────────────────────────
best_dataset = best_row['Dataset']
best_mname   = best_row['Model']
best_key     = f'{best_dataset}_{best_mname}'

best_model_obj, Xte_best = trained_models[best_key]
y_best_pred = (best_model_obj.predict(Xte_best, verbose=0).ravel() >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_best_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Benign (0)', 'Malware (1)'],
    yticklabels=['Benign (0)', 'Malware (1)'],
    linewidths=0.5, ax=ax, annot_kws={'size': 14}
)
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual',    fontsize=12, fontweight='bold')
ax.set_title(
    f'Confusion Matrix — Best Model\n({best_dataset} | {best_mname})',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print('\nClassification Report (Best Model):')
print(classification_report(y_test, y_best_pred, target_names=['Benign', 'Malware']))

In [ ]:
# ── Best Model Architecture Summary ──────────────────────────────────────────
print('='*60)
print('  BEST MODEL — Final Summary')
print('='*60)
print(f'  Dataset Version : {best_row["Dataset"]}')
print(f'  Model           : {best_row["Model"]}')
print(f'  Epochs Run      : {best_row["Epochs Run"]}')
print(f'  Training Acc    : {best_row["Train Acc"]:.4f}')
print(f'  Test Accuracy   : {best_row["Test Acc"]:.4f}')
print(f'  Precision       : {best_row["Precision"]:.4f}')
print(f'  Recall          : {best_row["Recall"]:.4f}')
print(f'  F1-Score        : {best_row["F1-Score"]:.4f}')
print('='*60)
best_model_obj.summary()

---
## Discussion of Results

### 1. Raw vs. Normalized Dataset

The Android Permission dataset is primarily composed of **binary flag columns** (0/1) indicating whether a particular permission is requested, with a small number of numeric columns (`Rating`, `Number of ratings`, `Price`, `Dangerous permissions count`, `Safe permissions count`).

- **Raw Dataset:** Most features are already bounded in {0, 1}, so networks can learn reasonable representations. However, the few large-scale columns (e.g., `Number of ratings` reaching millions) can create gradient imbalances, making optimization less stable.

- **Normalized Dataset (MinMaxScaler to [0,1]):** Brings all features onto the same scale, eliminating the dominance of large-valued columns. This leads to:
  - Faster convergence (fewer epochs needed)
  - More stable training curves
  - Generally equal or better generalization performance

---

### 2. Model Complexity Trade-offs

| Model | Hidden Layers | Parameters | Observation |
|---|---|---|---|
| Model 1 | 6 (128→64→32→16→8→1) | Most | High capacity, risk of overfitting |
| Model 2 | 5 (64→32→16→8→1)    | High | Good balance of capacity & speed |
| Model 3 | 4 (32→16→8→1)       | Medium | Lean; fast convergence |
| Model 4 | 3 (16→8→1)          | Least | Simplest; may underfit complex patterns |

**Observations:**
- Deeper models (Model 1 & 2) can capture more complex boundaries but take longer to train.
- Shallower models (Model 3 & 4) converge faster and may generalize equally well when features are primarily binary.
- **BatchNormalization + Dropout(0.2)** regularizes all models and limits overfitting.
- **Class weights** compensate for the Benign:Malware imbalance, ensuring Malware samples are not overwhelmed in gradient updates.

---

### 3. Key Takeaway

> For binary permission-based malware detection, **normalized inputs consistently produce more stable training** and competitive metrics. The best model (identified above) maximizes the **F1-Score**, which balances Precision and Recall — both critical in a security context where **missing malware (False Negative)** or **wrongly flagging benign apps (False Positive)** both carry significant real-world consequences. A mid-complexity architecture often outperforms both the deepest and shallowest models in this setting.